## 2.1 理论计算题
给定一个字符序列 "ababc"，假设采用一阶马尔可夫模型（即 p(xt
|xt−1)），
使用拉普拉斯平滑（加 1 平滑）估计以下条件概率：
1. p(’a’ | ’b’)
2. p(’c’ | ’b’)
（词汇表为 {'a','b','c'}，计算时考虑所有可能转移，包括未出现的情况。

给定字符序列 "ababc"，词汇表为 {'a','b','c'}。

首先，我们需要统计字符转移的频次。序列 "ababc" 中的转移有：
1. a → b
2. b → a
3. a → b
4. b → c

统计结果：
- a → b: 2次
- b → a: 1次
- b → c: 1次
- a → a: 0次
- a → c: 0次
- b → b: 0次
- c → a: 0次
- c → b: 0次
- c → c: 0次

使用拉普拉斯平滑（加1平滑），我们对每个转移计数都加1，并除以规范化因子（前一个字符的总出现次数 + 词汇表大小）。

1. 计算 p('a' | 'b')：

原始计数：b → a 出现1次
拉普拉斯平滑后：1 + 1 = 2
规范化因子：b的总出现次数 + 词汇表大小 = 3 + 3 = 6
p('a' | 'b') = 2/6 = 1/3 ≈ 0.333

2. 计算 p('c' | 'b')：

原始计数：b → c 出现1次
拉普拉斯平滑后：1 + 1 = 2
规范化因子：b的总出现次数 + 词汇表大小 = 3 + 3 = 6
p('c' | 'b') = 2/6 = 1/3 ≈ 0.333

所以，答案是：
1. p('a' | 'b') = 1/3 ≈ 0.333
2. p('c' | 'b') = 1/3 ≈ 0.333

In [1]:
# 2.2 编程题
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 只保留字母和空格
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    vocabulary = {word: idx for idx, (word, _) in enumerate(word_counts.most_common())}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(words) - n):
        # 特征：当前窗口中的n个词
        feature = words[i:i+n]
        # 标签：窗口后的下一个词
        label = words[i+n] if i+n < len(words) else None
        
        features.append(feature)
        if label is not None:  # 忽略无后续词的情况
            labels.append(label)
    
    return vocabulary, (features, labels)


In [2]:
text = "The time machine"
vocabulary, (features, labels) = preprocess_text(text, 2)
print("词汇表:", vocabulary)
print("特征:", features)
print("标签:", labels)


词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time']]
标签: ['machine']


## 3.1 理论计算题
考虑一个线性 RNN（无偏置），定义为 ht = Whhht−1 + Whxxt，输出
ot = Wohht。假设损失函数为平方损失 L =
1
2
∑T
t=1(ot − yt)
2。推导损失对
权重 Whh 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明
梯度消失或爆炸的条件。

## 
给定：
- 线性RNN：h_t = W_hh * h_{t-1} + W_hx * x_t
- 输出：o_t = W_oh * h_t
- 损失函数：L = (1/2) * Σ_{t=1}^T (o_t - y_t)^2

首先，我们需要计算损失对Whh的梯度∂L/∂Whh。

1. 根据链式法则：
∂L/∂Whh = Σ_{t=1}^T ∂L/∂h_t * ∂h_t/∂Whh

2. 计算∂L/∂h_t：
∂L/∂h_t = ∂L/∂o_t * ∂o_t/∂h_t
        = (o_t - y_t) * W_oh^T

3. 计算∂h_t/∂Whh：
h_t = W_hh * h_{t-1} + W_hx * x_t
∂h_t/∂Whh = ∂(W_hh * h_{t-1})/∂Whh + ∂(W_hx * x_t)/∂Whh
          = ∂(W_hh * h_{t-1})/∂Whh  (因为第二项不依赖Whh)
          = Σ_{k=1}^t ∂h_t/∂h_k * ∂h_k/∂Whh

4. 计算∂h_t/∂h_{t-1}：
∂h_t/∂h_{t-1} = W_hh

通过递归展开，我们可以得到：
∂h_t/∂h_{t-1} = W_hh
∂h_t/∂h_{t-2} = W_hh * W_hh
...
∂h_t/∂h_0 = W_hh^t

因此：
∂h_t/∂Whh = Σ_{k=1}^t W_hh^{t-k} * ∂h_k/∂Whh

5. 综合以上结果：
∂L/∂Whh = Σ_{t=1}^T (o_t - y_t) * W_oh^T * [Σ_{k=1}^t W_hh^{t-k} * ∂h_k/∂Whh]

梯度消失或爆炸的条件：

1. 梯度消失：
当|λ_max(W_hh)| < 1时（λ_max是Whh的最大特征值），随着时间步t的增加，W_hh^t会趋向于0，导致梯度随时间反向传播时逐渐减小。这在处理长序列时会导致早期时间步的梯度几乎为零，使得模型难以学习长距离依赖关系。

2. 梯度爆炸：
当|λ_max(W_hh)| > 1时，随着时间步t的增加，W_hh^t会指数级增长，导致梯度变得非常大。这会使训练过程不稳定，参数更新幅度过大，难以收敛。

In [7]:
# 3.2 编程题
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN前向传播
    
    参数:
    x_t -- 当前输入，形状为(batch_size, input_size)
    h_prev -- 上一隐藏状态，形状为(batch_size, hidden_size)
    W_hx -- 输入到隐藏状态的权重，形状为(hidden_size, input_size)
    W_hh -- 隐藏状态到隐藏状态的权重，形状为(hidden_size, hidden_size)
    b_h -- 隐藏状态的偏置，形状为(hidden_size,)
    
    返回:
    h_t -- 当前隐藏状态，形状为(batch_size, hidden_size)
    cache -- 缓存中间变量，用于反向传播
    """
    # 计算当前隐藏状态
    z = np.dot(x_t, W_hx.T) + np.dot(h_prev, W_hh.T) + b_h
    h_t = np.tanh(z)
    
    # 缓存中间变量
    cache = (x_t, h_prev, z, h_t)
    
    return h_t, cache

def rnn_backward(dh_next, cache):
    """
    RNN反向传播
    
    参数:
    dh_next -- 损失对当前隐藏状态的梯度，形状为(batch_size, hidden_size)
    cache -- 前向传播时缓存的中间变量
    
    返回:
    dx_t -- 损失对输入的梯度，形状为(batch_size, input_size)
    dh_prev -- 损失对上一隐藏状态的梯度，形状为(batch_size, hidden_size)
    dW_hx -- 损失对W_hx的梯度，形状为(hidden_size, input_size)
    dW_hh -- 损失对W_hh的梯度，形状为(hidden_size, hidden_size)
    db_h -- 损失对b_h的梯度，形状为(hidden_size,)
    """
    x_t, h_prev, z, h_t = cache
    
    # 计算tanh的梯度
    dz = dh_next * (1 - h_t ** 2)
    
    # 计算各参数的梯度
    dW_hx = np.dot(dz.T, x_t)
    dW_hh = np.dot(dz.T, h_prev)
    db_h = np.sum(dz, axis=0)
    
    # 计算输入和上一隐藏状态的梯度
    dx_t = np.dot(dz, W_hx)
    dh_prev = np.dot(dz, W_hh)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 参数设置
batch_size = 2
input_size = 3
hidden_size = 4

# 初始化参数
W_hx = np.random.randn(hidden_size, input_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size)

# 初始化输入和隐藏状态
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)

# 前向传播
h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)

# 假设的下游梯度
dh_next = np.random.randn(batch_size, hidden_size)

# 反向传播
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)

# 打印结果
print("=== 前向传播结果 ===")
print("输入 x_t:")
print(x_t)
print("\n上一隐藏状态 h_prev:")
print(h_prev)
print("\n当前隐藏状态 h_t:")
print(h_t)
print("\n=== 反向传播结果 ===")
print("输入梯度 dx_t:")
print(dx_t)
print("\n上一隐藏状态梯度 dh_prev:")
print(dh_prev)
print("\n权重梯度 dW_hx:")
print(dW_hx)
print("\n权重梯度 dW_hh:")
print(dW_hh)
print("\n偏置梯度 db_h:")
print(db_h)


=== 前向传播结果 ===
输入 x_t:
[[ 1.73042911  0.21943494  0.32750872]
 [-1.13413298  0.15598828 -1.42827432]]

上一隐藏状态 h_prev:
[[-1.1751784  -0.01671669  0.36906568 -0.38454728]
 [ 0.86176018  0.98630786  1.22243263  0.3744729 ]]

当前隐藏状态 h_t:
[[ 0.14635561 -0.34723802  0.74384508 -0.02328816]
 [-0.48019927 -0.94218285  0.04671449  0.99890647]]

=== 反向传播结果 ===
输入梯度 dx_t:
[[-0.42966777  1.60062476  2.40904275]
 [ 1.19686475  2.00442301 -0.2862514 ]]

上一隐藏状态梯度 dh_prev:
[[ 0.43042981  2.08805559  1.04473152  0.2120856 ]
 [ 1.60409317  1.8673759  -0.62734764 -0.49710099]]

权重梯度 dW_hx:
[[-0.87322647 -0.3453818   0.78458876]
 [ 1.13664255  0.11106986  0.34898277]
 [ 1.16878655 -0.01575672  0.88496247]
 [ 0.61692449  0.07799668  0.11771413]]

权重梯度 dW_hh:
[[ 0.52138393 -0.75493709 -1.33231276  0.09822591]
 [-0.78201945 -0.11855699  0.08091556 -0.26609651]
 [-0.84381845 -0.5447286  -0.5515849  -0.32669922]
 [-0.41904093 -0.00672505  0.13042862 -0.13719307]]

偏置梯度 db_h:
[-1.80025676  0.47427271 -0.2299399

## 4.1 理论计算题
假设一个深度双向 RNN，有 L 层，每层隐藏单元数为 H，输入维度为
D，输出维度为 O（仅考虑最后输出层）。计算该模型的参数总数（包括所有
全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达
式。

## 
对于一个深度双向RNN，我们需要考虑以下几点：

1. 每个双向RNN单元实际上包含两个方向的RNN（前向和后向）
2. 对于L层网络，每一层都有前向和后向的RNN单元
3. 每个RNN单元的参数包括：
   - W_hx: 输入到隐藏状态的权重
   - W_hh: 隐藏状态到隐藏状态的权重
   - b_h: 隐藏状态的偏置

让我们逐层计算参数：

对于第l层（l从1到L）：
1. 前向RNN单元的参数：
   - W_hx: 形状为(H, D)（如果是第一层）或(H, H)（如果是其他层）
   - W_hh: 形状为(H, H)
   - b_h: 形状为(H,)
   - 参数总数：H×D + H×H + H（第一层）或 H×H + H×H + H（其他层）

2. 后向RNN单元的参数：
   - 与前向RNN单元相同
   - 参数总数：H×D + H×H + H（第一层）或 H×H + H×H + H（其他层）

因此，对于整个网络：
- 第一层：
  - 前向：H×D + H×H + H
  - 后向：H×D + H×H + H
  - 总计：2×(H×D + H×H + H)

- 其他层（l=2到L）：
  - 前向：H×H + H×H + H
  - 后向：H×H + H×H + H
  - 总计：2×(2×H×H + H)

所以整个网络的参数总数为：
总参数 = 2×(H×D + H×H + H) + (L-1)×2×(2×H×H + H)

简化表达式：
总参数 = 2H×D + 4H×H + 2H + (L-1)×(4H×H + 2H)
      = 2H×D + 4L×H×H + 2H

因此，深度双向RNN的参数总数表达式为：
**总参数 = 2H×D + 4L×H×H + 2H**


In [15]:
# 4.2 编程题
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        """
        双向RNN编码器
        
        参数:
        input_dim -- 输入维度
        hidden_dim -- 隐藏层维度
        num_layers -- RNN层数，默认为1
        """
        super(BidirectionalRNNEncoder, self).__init__()
        self.rnn = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,  # 输入形状为(seq_len, batch, input_dim)
            bidirectional=True
        )
    
    def forward(self, x):
        """
        前向传播
        
        参数:
        x -- 输入序列，形状为(seq_len, batch, input_dim)
        
        返回:
        time_step_outputs -- 每个时间步的拼接隐藏状态，形状为(seq_len, batch, 2*hidden_dim)
        final_hidden -- 最终时间步的拼接隐藏状态，形状为(batch, 2*hidden_dim)，作为序列表示
        """
        # 前向传播
        outputs, (hidden, cell) = self.rnn(x)
        
        # outputs形状为(seq_len, batch, 2*hidden_dim)
        # 已经包含了每个时间步的前向和后向隐藏状态的拼接
        
        # 获取最终时间步的拼接隐藏状态
        # hidden形状为(num_layers*2, batch, hidden_dim)
        # 取最后一层的前向和后向隐藏状态并拼接
        final_hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        
        return outputs, final_hidden

# 使用示例
if __name__ == "__main__":
    # 参数设置
    seq_len = 5
    batch_size = 3
    input_dim = 4
    hidden_dim = 6
    num_layers = 1

    # 创建输入数据
    x = torch.randn(seq_len, batch_size, input_dim)

    # 创建编码器实例
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers)

    # 前向传播
    time_step_outputs, final_hidden = encoder(x)

    # 打印结果
    print("=== 输入信息 ===")
    print(f"输入形状: {x.shape}")
    print(f"序列长度: {seq_len}, 批次大小: {batch_size}, 输入维度: {input_dim}")
    print("\n=== 输出信息 ===")
    print("每个时间步的拼接隐藏状态形状:", time_step_outputs.shape)
    print("最终时间步的拼接隐藏状态形状:", final_hidden.shape)



=== 输入信息 ===
输入形状: torch.Size([5, 3, 4])
序列长度: 5, 批次大小: 3, 输入维度: 4

=== 输出信息 ===
每个时间步的拼接隐藏状态形状: torch.Size([5, 3, 12])
最终时间步的拼接隐藏状态形状: torch.Size([3, 12])


## 5.1 理论计算题
在 Skip-gram 模型中，给定中心词 wc 和上下文词 wo，使用负采样（采
样 K 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪
声分布中采样负样本。假设词向量为 vc, uo，负样本词向量为 unk，写出完
整的目标函数。

## 
Skip-gram模型负采样损失函数推导

在Skip-gram模型中，给定中心词w_c和上下文词w_o，我们希望最大化它们同时出现的概率。使用负采样时，我们只考虑一个正样本（w_o）和K个负样本（从噪声分布中采样）。

#### 1. 概率模型

对于正样本(w_c, w_o)，我们使用sigmoid函数来建模它们的共现概率：
P(w_o | w_c) = σ(v_c^T u_o) = 1 / (1 + exp(-v_c^T u_o))

其中：
- v_c是中心词w_c的词向量
- u_o是上下文词w_o的词向量
- σ是sigmoid函数

对于负样本(w_c, ō)，我们希望它们的共现概率尽可能小：
P(ō | w_c) = 1 - σ(v_c^T ū_k) = σ(-v_c^T ū_k)

#### 2. 损失函数推导

负采样的目标函数是对数似然函数：

L = -[log P(w_o | w_c) + Σ_{k=1}^K log P(ō_k | w_c)]

将概率表达式代入：

L = -[log σ(v_c^T u_o) + Σ_{k=1}^K log σ(-v_c^T ū_k)]

进一步展开sigmoid函数：

L = -[log(1/(1+exp(-v_c^T u_o))) + Σ_{k=1}^K log(1/(1+exp(v_c^T ū_k)))]

简化对数表达式：

L = -[-log(1+exp(-v_c^T u_o)) + Σ_{k=1}^K -log(1+exp(v_c^T ū_k))]

最终得到负采样的损失函数：

L = log(1+exp(-v_c^T u_o)) + Σ_{k=1}^K log(1+exp(v_c^T ū_k))

#### 3. 负样本采样

从噪声分布中采样负样本通常使用以下方法：

1. **噪声分布**：通常使用词频的3/4次方作为噪声分布：
   P(n) = count(n)^3/4 / Σ_w count(w)^3/4

2. **采样方法**：
   - 预先计算每个词的采样概率
   - 使用Alias方法或直接采样来高效采样K个负样本

#### 4. 完整目标函数

完整的负采样Skip-gram目标函数为：

J = (1/T) Σ_{t=1}^T [log(1+exp(-v_c^T u_o)) + Σ_{k=1}^K log(1+exp(v_c^T ū_k))]

其中：
- T是训练样本总数
- v_c是中心词w_c的词向量
- u_o是正样本上下文词w_o的词向量
- ū_k是第k个负样本词ō_k的词向量
- K是负样本数量


In [16]:
# 5.2 编程题
import numpy as np

def cbow_forward_loss(context_indices, target_index, W, W_out):
    """
    CBOW模型前向传播和损失计算
    
    参数:
    context_indices -- 一批上下文词的索引列表，形状为(batch_size, context_size)
    target_index -- 目标中心词索引，形状为(batch_size,)
    W -- 输入权重矩阵（嵌入矩阵），形状为(V, d)
    W_out -- 输出权重矩阵，形状为(d, V)
    
    返回:
    loss -- 交叉熵损失值
    """
    batch_size = context_indices.shape[0]
    context_size = context_indices.shape[1]
    V, d = W.shape
    
    # 1. 获取上下文词的嵌入
    # 形状: (batch_size, context_size, d)
    context_embeddings = W[context_indices]
    
    # 2. 计算平均上下文向量作为隐藏层
    # 形状: (batch_size, d)
    hidden = np.mean(context_embeddings, axis=1)
    
    # 3. 计算输出层的logits
    # 形状: (batch_size, V)
    logits = np.dot(hidden, W_out)
    
    # 4. 计算softmax概率
    # 数值稳定性处理：减去最大值
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    
    # 5. 计算交叉熵损失
    # 只考虑目标词的概率
    batch_probs = probs[np.arange(batch_size), target_index]
    loss = -np.mean(np.log(batch_probs + 1e-10))  # 添加小常数避免log(0)
    
    return loss

# 使用示例
if __name__ == "__main__":
    # 参数设置
    batch_size = 2
    context_size = 4
    V = 1000  # 词汇表大小
    d = 300   # 嵌入维度
    
    # 创建随机权重矩阵
    np.random.seed(42)
    W = np.random.randn(V, d) * 0.01  # 输入权重矩阵
    W_out = np.random.randn(d, V) * 0.01  # 输出权重矩阵
    
    # 创建随机输入数据
    context_indices = np.random.randint(0, V, size=(batch_size, context_size))
    target_index = np.random.randint(0, V, size=batch_size)
    
    # 计算损失
    loss = cbow_forward_loss(context_indices, target_index, W, W_out)
    
    # 打印结果
    print("=== CBOW模型前向传播和损失计算 ===")
    print(f"上下文词索引形状: {context_indices.shape}")
    print(f"目标词索引形状: {target_index.shape}")
    print(f"输入权重矩阵W形状: {W.shape}")
    print(f"输出权重矩阵W_out形状: {W_out.shape}")
    print(f"计算得到的损失值: {loss:.4f}")


=== CBOW模型前向传播和损失计算 ===
上下文词索引形状: (2, 4)
目标词索引形状: (2,)
输入权重矩阵W形状: (1000, 300)
输出权重矩阵W_out形状: (300, 1000)
计算得到的损失值: 6.9083


## 6.1 理论计算题
给定查询矩阵 Q ∈ R
2×4，键矩阵 K ∈ R
3×4，值矩阵 V ∈ R
3×5。计算
缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩
阵，再 softmax，再加权求和）。使用 score = QKT /
√
dk（dk = 4）。可以只
列出数值计算过程（用符号或具体数值）

## 
给定：
- 查询矩阵 Q ∈ R²×⁴
- 键矩阵 K ∈ R³×⁴
- 值矩阵 V ∈ R³×⁵

假设这些矩阵的具体值为：
```
Q = [1 2 3 4]
    [5 6 7 8]

K = [1 1 1 1]
    [2 2 2 2]
    [3 3 3 3]

V = [1 2 3 4 5]
    [6 7 8 9 10]
    [11 12 13 14 15]
```

计算步骤如下：

1. 计算得分矩阵：
```
QK^T = [1 2 3 4]   [1 2 3]   = [1×1+2×1+3×1+4×1  1×2+2×2+3×2+4×2  1×3+2×3+3×3+4×3]
       [5 6 7 8]   [1 2 3]     [5×1+6×1+7×1+8×1  5×2+6×2+7×2+8×2  5×3+6×3+7×3+8×3]
                   [1 2 3]
    
      = [10 20 30]
        [26 52 78]
```

2. 缩放得分（dk = 4，√dk = 2）：
```
QK^T/√dk = [10/2 20/2 30/2] = [5 10 15]
           [26/2 52/2 78/2]   [13 26 39]
```

3. 应用softmax：
对每一行应用softmax函数：

第一行：
```
exp(5) = 148.413
exp(10) = 22026.466
exp(15) = 3269017.372
sum = 3291081.251

softmax = [148.413/3291081.251  22026.466/3291081.251  3269017.372/3291081.251]
         = [4.51e-5  6.70e-3  0.9933]
```

第二行：
```
exp(13) = 442413.392
exp(26) = 1.957e11
exp(39) = 8.553e16
sum = 8.555e16

softmax = [442413.392/8.555e16  1.957e11/8.555e16  8.553e16/8.555e16]
         = [5.17e-12  2.29e-6  0.99999771]
```

4. 加权求和：
```
Attention = [4.51e-5  6.70e-3  0.9933]   [1  2  3  4  5]   = [0.9933×11 + 6.70e-3×6 + 4.51e-5×1
            [5.17e-12  2.29e-6  0.99999771] [6  7  8  9 10]     0.99999771×11 + 2.29e-6×6 + 5.17e-12×1
                                   [11 12 13 14 15]]
    
         = [11.003  12.003  13.003  14.003  15.003]
           [11.000  12.000  13.000  14.000  15.000]
```

因此，缩放点积注意力的输出矩阵为：
```
[11.003  12.003  13.003  14.003  15.003]
[11.000  12.000  13.000  14.000  15.000]
```

这个结果展示了注意力机制如何通过查询、键和值的相互作用，生成加权和的输出矩阵。在这个例子中，第一个查询向量主要关注第三个键向量，而第二个查询向量几乎完全关注第三个键向量。

In [20]:
# 6.2 编程题
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        多头注意力模块
        :param d_model: 模型维度（必须能被 num_heads 整除）
        :param num_heads: 头数
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads   # 每个头的 K 维度
        self.d_v = d_model // num_heads   # 每个头的 V 维度（通常与 d_k 相同）

        # 线性投影层（无偏置，也可根据需求添加）
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        """
        前向传播
        :param X: 输入张量，形状 (seq_len, batch, d_model)
        :return: 输出张量，形状 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.size()

        # 1. 线性投影得到 Q, K, V（形状均为 (seq_len, batch, d_model)）
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        # 2. 拆分多头：将最后一维拆成 (num_heads, d_k) 或 (num_heads, d_v)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v)

        # 3. 调整维度顺序以便并行计算：将 batch 和 head 维度提前
        #    最终形状：(batch, num_heads, seq_len, d_k) 或 (batch, num_heads, seq_len, d_v)
        Q = Q.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)

        # 4. 缩放点积注意力
        #    scores = Q * K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq_len, seq_len)
        context = torch.matmul(attn_weights, V)   # (batch, num_heads, seq_len, d_v)

        # 5. 拼接所有头的输出
        #    先恢复顺序 (seq_len, batch, num_heads, d_v)，再合并最后一维
        context = context.permute(2, 0, 1, 3).contiguous()  # (seq_len, batch, num_heads, d_v)
        context = context.view(seq_len, batch, self.d_model) # (seq_len, batch, d_model)

        # 6. 最终线性层
        output = self.W_o(context)  # (seq_len, batch, d_model)

        return output


# ========== 测试代码 ==========
if __name__ == "__main__":
    seq_len, batch, d_model = 10, 4, 4
    num_heads = 2
    X = torch.randn(seq_len, batch, d_model)

    mha = MultiHeadAttention(d_model, num_heads)
    out = mha(X)

    print("输入形状:", X.shape)      # torch.Size([10, 4, 4])
    print("输出形状:", out.shape)    # torch.Size([10, 4, 4])

输入形状: torch.Size([10, 4, 4])
输出形状: torch.Size([10, 4, 4])
